# 00 · Toolchain and fidelity test

> **Placeholder:** no code yet. Filled in during **Phase 3** of `docs/EXPERIMENT_PLAN.md`.

## Purpose
Prove that a model quantized in PyTorch keeps its quantization through conversion to `.tflite` and runs identically on the ESP32-S3, before any real training is done.

## Experiments covered
- Phase 3 gate (no experiment ID); the fidelity metric of §7

## Inputs
- A tiny convolutional model defined here
- Random and real (B-cal) input images

## Outputs
- `artifacts/toolchain/tiny_ptq.tflite`, `tiny_qat.tflite`
- `results/fidelity_*.json` (output differences per stage)

## Depends on
- PyTorch with PT2E quantization (torchao)
- `ai-edge-torch` / LiteRT converter; `onnx2tf` as fallback
- Phase 4 TFLite Micro backend for the on-device step

## Pass criterion
Mean absolute difference between PyTorch fake-quant, TFLite on the PC and TFLite on the S3 is ≤ 1 quantization step of each output tensor.

## 1. Environment

Pin versions of torch, torchao, ai-edge-torch/LiteRT, onnx, onnx2tf and tflite-runtime. Record them here.

## 2. Tiny reference model

A few conv + ReLU layers with a detection-like output, small enough to run anywhere.

## 3. PT2E post-training quantization

Per-channel int8 weights, per-tensor int8 activations, calibrated on B-cal frames.

## 4. PT2E quantization-aware training

A few hundred QAT steps on the same model, to exercise the QAT export path.

## 5. Convert to .tflite

`ai-edge-torch` first; if it fails or loses the quantization, ONNX → `onnx2tf` with full-int8.

## 6. Inspect the .tflite

Operators, tensor dtypes, scales/zero-points. Confirm the trained quantization parameters survived conversion.

## 7. PC fidelity

Same 100 inputs through PyTorch fake-quant and the TFLite interpreter; compare outputs.

## 8. Device fidelity

Pack, flash, run the same inputs on the S3 through the TFLM backend; compare with the PC.

## 9. Gate decision

Pass/fail against the criterion above; choose the conversion path for all later notebooks.